In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

## Install libraries

In [ ]:
# !pip install -q youtube-transcript-api langchain-community langchain-openai \n#                faiss-cpu tiktoken python-dotenv
# !pip install --upgrade youtube-transcript-api  # fix ParseError from YouTube API changes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.9/433.9 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.9 MB/s eta 0:00:00


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

## Step 1a - Indexing (Document Ingestion)

In [ ]:
import xml.etree.ElementTree as ET

video_id = "Gfr50f6ZBvo"
transcript = None

try:
    api = YouTubeTranscriptApi()
    raw = api.fetch(video_id, languages=["en"])
    transcript = " ".join(chunk.text for chunk in raw)
    print(transcript[:500])

except TranscriptsDisabled:
    print("Captions are disabled for this video.")
except NoTranscriptFound:
    print("No English transcript — try listing available languages with api.list(video_id)")
except ET.ParseError:
    print("Empty XML response — run: pip install --upgrade youtube-transcript-api")
except Exception as e:
    print(f"{type(e).__name__}: {e}")

In [5]:
transcript_list

[{'text': 'the following is a conversation with',
  'start': 0.08,
  'duration': 3.44},
 {'text': 'demus hasabis', 'start': 1.76, 'duration': 4.96},
 {'text': 'ceo and co-founder of deepmind', 'start': 3.52, 'duration': 5.119},
 {'text': 'a company that has published and builds',
  'start': 6.72,
  'duration': 4.48},
 {'text': 'some of the most incredible artificial',
  'start': 8.639,
  'duration': 4.561},
 {'text': 'intelligence systems in the history of',
  'start': 11.2,
  'duration': 4.8},
 {'text': 'computing including alfred zero that',
  'start': 13.2,
  'duration': 3.68},
 {'text': 'learned', 'start': 16.0, 'duration': 2.96},
 {'text': 'all by itself to play the game of gold',
  'start': 16.88,
  'duration': 4.559},
 {'text': 'better than any human in the world and',
  'start': 18.96,
  'duration': 5.6},
 {'text': 'alpha fold two that solved protein',
  'start': 21.439,
  'duration': 4.241},
 {'text': 'folding', 'start': 24.56, 'duration': 4.16},
 {'text': 'both tasks consider

## Step 1b - Indexing (Text Splitting)

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [7]:
len(chunks)

168

In [9]:
chunks[100]

Document(metadata={}, page_content="and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask w

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [10]:
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

C:\Users\debab\AppData\Local\Temp\ipykernel_25108\1938714538.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
c:\Users\debab\Desktop\IIT+SELF LEARNING\CODING\LangChain Model\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
vector_store.index_to_docstore_id

{0: 'e9e624c0-71ad-4e88-97ae-e851de42e43c',
 1: 'dd1ef780-82b1-4ed1-a447-e71fbc6796fc',
 2: 'ee2cff86-62a6-46f6-81d5-15136e8d667d',
 3: '1f1a60c3-aac4-4119-9da4-19bac902857f',
 4: '8b2dc74c-e118-4b72-bbd8-831e23c7ebcc',
 5: '685fef95-507e-4a2a-a409-cdc062513af7',
 6: 'b9879d04-e9a3-4257-b3b7-4ee1d858e7da',
 7: '25ee7189-9301-4798-891e-d94358fb8467',
 8: '66e27cee-e974-4d80-b64e-de6415868a5f',
 9: '2fdb10d4-57b6-48b2-8211-fe81dea0b83d',
 10: '830fe6da-4205-456e-b221-bf5a265e9510',
 11: 'ab0ec1a9-8ddf-48ed-8ef3-6d0b19fccbbc',
 12: '25e98bb1-c3e4-41f0-9ddf-29437d289e18',
 13: 'adcef6f7-4ffc-4aaf-8129-c021c53b68f3',
 14: '7443b4f0-4d57-42e6-bc51-1db03650f7c2',
 15: '9527914a-b3f8-42e2-8cb7-aa3e11a2b9db',
 16: '27e0cd91-69a7-44fb-917f-5f89641ef774',
 17: 'd4dd2348-837d-42c9-91bb-a726ad6eda9e',
 18: '1cb17333-fadf-423b-a2ba-fb539e097a2c',
 19: 'ab8a96b9-5a26-4a04-a5d4-ba6e3b43d1cb',
 20: '34dd507e-e6ac-420e-a2dc-5c791cf72d5a',
 21: 'f9eefdf5-346f-439f-95f7-61fbfe6018a0',
 22: 'a127a6c1-4ea4-

In [12]:
vector_store.get_by_ids(['e9e624c0-71ad-4e88-97ae-e851de42e43c'])

[Document(id='e9e624c0-71ad-4e88-97ae-e851de42e43c', metadata={}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal qu

## Step 2 - Retrieval

In [13]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [14]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E83C751B50>, search_kwargs={'k': 4})

In [15]:
retriever.invoke('What is deepmind')

[Document(id='cbcfe4c1-2c0d-4c7e-98ff-bf8bd08cae07', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

## Step 3 - Augmentation

In [50]:
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
llm = HuggingFaceEndpoint(
    repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation"
    # max_new_tokens=1000,
    # temperature=0.7
)
model = ChatHuggingFace(llm=llm)

In [39]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [32]:
question          = "is the topic of Alien discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [33]:
retrieved_docs

[Document(id='6db4b282-dc2e-4a47-aac3-06d9240dbae3', metadata={}, page_content="space age we should have heard a cacophony of voices we should have joined that cacophony of voices and what we did we opened our ears and we heard nothing and many people who argue that there are aliens would say well we haven't really done exhaustive search yet and maybe we're looking in the wrong bands and and we've got the wrong devices and we wouldn't notice what an alien form was like to be so different to what we're used to but you know i'm not i don't really buy that that it shouldn't be as difficult as that like we i think we've searched enough there should be if it were everywhere if it was it should be everywhere we should see dyson's fears being put up sun's blinking in and out you know there should be a lot of evidence for those things and then there are other people argue well the sort of safari view of like well we're a primitive species still because we're not space faring yet and and and we

In [40]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"space age we should have heard a cacophony of voices we should have joined that cacophony of voices and what we did we opened our ears and we heard nothing and many people who argue that there are aliens would say well we haven't really done exhaustive search yet and maybe we're looking in the wrong bands and and we've got the wrong devices and we wouldn't notice what an alien form was like to be so different to what we're used to but you know i'm not i don't really buy that that it shouldn't be as difficult as that like we i think we've searched enough there should be if it were everywhere if it was it should be everywhere we should see dyson's fears being put up sun's blinking in and out you know there should be a lot of evidence for those things and then there are other people argue well the sort of safari view of like well we're a primitive species still because we're not space faring yet and and and we're you know there's some kind of globe like universal rule not to interfere\n\

In [41]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [42]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      space age we should have heard a cacophony of voices we should have joined that cacophony of voices and what we did we opened our ears and we heard nothing and many people who argue that there are aliens would say well we haven't really done exhaustive search yet and maybe we're looking in the wrong bands and and we've got the wrong devices and we wouldn't notice what an alien form was like to be so different to what we're used to but you know i'm not i don't really buy that that it shouldn't be as difficult as that like we i think we've searched enough there should be if it were everywhere if it was it should be everywhere we should see dyson's fears being put up sun's blinking in and out you know there should be a lot of evidence for those things and then there are other people argue well the sor

## Step 4 - Generation

In [51]:
answer = model.invoke(final_prompt)
print(answer.content)

Yes, the topic of Alien discussed in this video is the answer to the question, "Can humans contact other celestial beings or might we just be alone in the universe?" Some possible answers include that humans are alone in the galaxy, there are no other intelligent life forms, there are advanced civilizations out there, and hugging our phones or built-in connectivity is just the beginning. Partisans on both sides of the question debate different possibilities and evidence. It's hard to know with certainty since there is still so much to learn about the universe.


## Building a Chain

In [52]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [53]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [54]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [56]:
parallel_chain.invoke('who is Demis')

{'context': "to get world peace because there's also other corrupting things like wanting power over people and this kind of stuff which is not necessarily satisfied by by just abundance but i think it will help um and i think uh but i think ultimately ai is not going to be run by any one person or one organization i think it should belong to the world belong to humanity um and i think maybe many there'll be many ways this will happen and ultimately um everybody should have a say in that do you have advice for uh young people in high school and college maybe um if they're interested in ai or interested in having a big impact on the world what they should do to have a career they can be proud of her to have a life they can be proud of i love giving talks to the next generation what i say to them is actually two things i i think the most important things to learn about and to find out about when you're when you're young is what are your true passions is first of all there's two things on

In [57]:
parser = StrOutputParser()

In [58]:
main_chain = parallel_chain | prompt | llm | parser

In [59]:
main_chain.invoke('Can you summarize the video')

c:\Users\debab\Desktop\IIT+SELF LEARNING\CODING\LangChain Model\venv\Lib\site-packages\huggingface_hub\utils\_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)


' Answer: Yes, the video summarizes the conversation with Demas Establish, a computer science professor at California Institute of Technology and an expert in the fields of quantum mechanics and computer science. He discusses the origins of consciousness, life, and gravity, and how they are all interconnected and explainable through the principles of quantum mechanics. The conversation covers the idea that there may be simpler or deeper explanations for these mysteries, as well as the importance of explaining complex topics simply and using language that is clear and concise.'